## Broken Timing Anchors Correction 

In [2]:
import importlib
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.preprocessing import StandardScaler

import src.paths as paths
import src.representations as representations
import src.metrics as metrics

importlib.reload(paths)
importlib.reload(representations)
importlib.reload(metrics)

from src.paths import get_project_paths
from src.representations import structural_campaign_summary
from src.metrics import c2st_rbf_paired

PATHS=get_project_paths()
DATA_DIR=PATHS["processed"]
RESULTS_DIR=PATHS["results"]

OUT_DIR=RESULTS_DIR/"week8"/"c2st_rbf_corrected"
OUT_DIR.mkdir(parents=True,exist_ok=True)

RBF_REPS=200
RBF_N=100
RBF_PERMUTATIONS=100
RBF_ALPHA=.05
RBF_SEVERITIES=[.10,.30,.60]

SAMPLE_BASE_SEED=42
NULL_SAMPLE_BASE_SEED=42000
TEST_BASE_SEED=10000

RBF_C=1.0
RBF_GAMMA="scale"

In [2]:
# load semantic embeddings
def load_embedding_family(prefix):
    return {
        "clean":np.load(DATA_DIR/f"{prefix}_clean.npy"),
        .10:np.load(DATA_DIR/f"{prefix}_10pct.npy"),
        .30:np.load(DATA_DIR/f"{prefix}_30pct.npy"),
        .60:np.load(DATA_DIR/f"{prefix}_60pct.npy")
    }

entity=load_embedding_family("entity_embeddings")
missing=load_embedding_family("missing_reference_embeddings")
over_polished=load_embedding_family("over_polished_embeddings")

for name,data in [
    ("Entity",entity),
    ("Missing",missing),
    ("Over-Polished",over_polished)
]:
    print(name,{k:v.shape for k,v in data.items()})

Entity {'clean': (300, 384), 0.1: (300, 384), 0.3: (300, 384), 0.6: (300, 384)}
Missing {'clean': (300, 384), 0.1: (300, 384), 0.3: (300, 384), 0.6: (300, 384)}
Over-Polished {'clean': (300, 384), 0.1: (300, 384), 0.3: (300, 384), 0.6: (300, 384)}


In [3]:
# load template tf idf 
template={
    "clean":sparse.load_npz(DATA_DIR/"template_tfidf_clean.npz"),
    .10:sparse.load_npz(DATA_DIR/"template_tfidf_10pct.npz"),
    .30:sparse.load_npz(DATA_DIR/"template_tfidf_30pct.npz"),
    .60:sparse.load_npz(DATA_DIR/"template_tfidf_60pct.npz")
}

print({k:v.shape for k,v in template.items()})

{'clean': (300, 5000), 0.1: (300, 5000), 0.3: (300, 5000), 0.6: (300, 5000)}


In [4]:
# reconstruct formal broken timing representation
campaigns=pd.read_csv(DATA_DIR/"clean_structured_campaigns.csv")
campaigns["campaign_id"]=campaigns["campaign_id"].astype(str)
campaigns["timestamp"]=pd.to_datetime(campaigns["timestamp"])

train_ids=pd.read_csv(
    DATA_DIR/"clean_structural_train_scaled.csv"
)["campaign_id"].astype(str).to_numpy()

eval_ids=pd.read_csv(
    DATA_DIR/"clean_structural_eval_scaled.csv"
)["campaign_id"].astype(str).to_numpy()

timing_files={
    "clean":"broken_timing_structural_clean.csv",
    .10:"broken_timing_structural_10pct.csv",
    .30:"broken_timing_structural_30pct.csv",
    .60:"broken_timing_structural_60pct.csv"
}

timing_raw={}

for key,name in timing_files.items():
    df=pd.read_csv(DATA_DIR/name)
    df["campaign_id"]=df["campaign_id"].astype(str)
    timing_raw[key]=df.set_index("campaign_id").loc[eval_ids].reset_index()

TIMING_FEATURES=[
    c for c in timing_raw["clean"].columns
    if c!="campaign_id"
]

train_campaigns=campaigns[
    campaigns["campaign_id"].isin(train_ids)
].copy()

train_summary=structural_campaign_summary(train_campaigns)
train_summary["campaign_id"]=train_summary["campaign_id"].astype(str)

train_summary=(
    train_summary
    .set_index("campaign_id")
    .loc[train_ids]
    .reset_index()
)

scaler=StandardScaler()
scaler.fit(train_summary[TIMING_FEATURES])

timing={
    key:scaler.transform(df[TIMING_FEATURES]).astype(np.float32)
    for key,df in timing_raw.items()
}

print({k:v.shape for k,v in timing.items()})

{'clean': (300, 13), 0.1: (300, 13), 0.3: (300, 13), 0.6: (300, 13)}


In [5]:
# validate the 5 scenario families 
SCENARIOS={
    "Template Repetition":{
        "clean":template["clean"],
        "conditions":{s:template[s] for s in RBF_SEVERITIES},
        "null_family":"template_tfidf"
    },
    "Broken Timing":{
        "clean":timing["clean"],
        "conditions":{s:timing[s] for s in RBF_SEVERITIES},
        "null_family":"timing_structural"
    },
    "Entity Inconsistency":{
        "clean":entity["clean"],
        "conditions":{s:entity[s] for s in RBF_SEVERITIES},
        "null_family":"campaign_minilm"
    },
    "Missing Cross-Channel Reference":{
        "clean":missing["clean"],
        "conditions":{s:missing[s] for s in RBF_SEVERITIES},
        "null_family":"campaign_minilm"
    },
    "Over-Polished Language":{
        "clean":over_polished["clean"],
        "conditions":{s:over_polished[s] for s in RBF_SEVERITIES},
        "null_family":"op_minilm"
    }
}

for scenario,cfg in SCENARIOS.items():
    n=cfg["clean"].shape[0]

    for severity,Q in cfg["conditions"].items():
        assert Q.shape==cfg["clean"].shape

    assert n>=2*RBF_N

    print(
        scenario,
        "| shape =",cfg["clean"].shape,
        "| null =",cfg["null_family"]
    )

Template Repetition | shape = (300, 5000) | null = template_tfidf
Broken Timing | shape = (300, 13) | null = timing_structural
Entity Inconsistency | shape = (300, 384) | null = campaign_minilm
Missing Cross-Channel Reference | shape = (300, 384) | null = campaign_minilm
Over-Polished Language | shape = (300, 384) | null = op_minilm


In [6]:
# smoke test the corrected rbf
cfg=SCENARIOS["Entity Inconsistency"]

P=cfg["clean"][:RBF_N]
Q=cfg["conditions"][.30][:RBF_N]

smoke=c2st_rbf_paired(
    P,
    Q,
    test_size=.5,
    n_permutations=RBF_PERMUTATIONS,
    seed=42,
    alpha=RBF_ALPHA,
    C=RBF_C,
    gamma=RBF_GAMMA
)

smoke

{'accuracy': 0.5,
 'p_value': 0.7920792079207921,
 'reject_H0': False,
 'n_pairs': 100,
 'n_train_pairs': 50,
 'n_test_pairs': 50,
 'n_test': 100,
 'n_permutations': 100,
 'seed': 42,
 'C': 1.0,
 'gamma': 'scale'}

In [7]:
# run 15 corrected anchors
power_records=[]

for scenario,cfg in SCENARIOS.items():

    print("\n",scenario)

    for rep in range(RBF_REPS):
        rng=np.random.default_rng(SAMPLE_BASE_SEED+rep)

        idx=rng.choice(
            cfg["clean"].shape[0],
            size=RBF_N,
            replace=False
        )

        P=cfg["clean"][idx]

        for severity in RBF_SEVERITIES:
            Q=cfg["conditions"][severity][idx]

            result=c2st_rbf_paired(
                P,
                Q,
                test_size=.5,
                n_permutations=RBF_PERMUTATIONS,
                seed=TEST_BASE_SEED+rep,
                alpha=RBF_ALPHA,
                C=RBF_C,
                gamma=RBF_GAMMA
            )

            power_records.append({
                "scenario":scenario,
                "severity":severity,
                "repetition":rep,
                "n":RBF_N,
                "accuracy":result["accuracy"],
                "p_value":result["p_value"],
                "reject":result["reject_H0"],
                "n_test":result["n_test"],
                "n_permutations":result["n_permutations"]
            })

        if (rep+1)%20==0:
            print(f"  {rep+1}/{RBF_REPS}")

rbf_power_runs=pd.DataFrame(power_records)

print("Rows:",len(rbf_power_runs))
print("Expected:",5*3*RBF_REPS)

assert len(rbf_power_runs)==3000


 Template Repetition
  20/200
  40/200
  60/200
  80/200
  100/200
  120/200
  140/200
  160/200
  180/200
  200/200

 Broken Timing
  20/200
  40/200
  60/200
  80/200
  100/200
  120/200
  140/200
  160/200
  180/200
  200/200

 Entity Inconsistency
  20/200
  40/200
  60/200
  80/200
  100/200
  120/200
  140/200
  160/200
  180/200
  200/200

 Missing Cross-Channel Reference
  20/200
  40/200
  60/200
  80/200
  100/200
  120/200
  140/200
  160/200
  180/200
  200/200

 Over-Polished Language
  20/200
  40/200
  60/200
  80/200
  100/200
  120/200
  140/200
  160/200
  180/200
  200/200
Rows: 3000
Expected: 3000


In [8]:
# build 4 independent clean-null families 
NULL_FAMILIES={
    "template_tfidf":template["clean"],
    "timing_structural":timing["clean"],
    "campaign_minilm":entity["clean"],
    "op_minilm":over_polished["clean"]
}

for name,X in NULL_FAMILIES.items():
    print(name,X.shape)
    assert X.shape[0]>=2*RBF_N

template_tfidf (300, 5000)
timing_structural (300, 13)
campaign_minilm (300, 384)
op_minilm (300, 384)


In [9]:
# run 4 correcte null calibrations
null_records=[]

for null_family,X in NULL_FAMILIES.items():

    print("\n",null_family)

    for rep in range(RBF_REPS):
        rng=np.random.default_rng(
            NULL_SAMPLE_BASE_SEED+rep
        )

        selected=rng.choice(
            X.shape[0],
            size=2*RBF_N,
            replace=False
        )

        P=X[selected[:RBF_N]]
        Q=X[selected[RBF_N:]]

        result=c2st_rbf_paired(
            P,
            Q,
            test_size=.5,
            n_permutations=RBF_PERMUTATIONS,
            seed=TEST_BASE_SEED+rep,
            alpha=RBF_ALPHA,
            C=RBF_C,
            gamma=RBF_GAMMA
        )

        null_records.append({
            "null_family":null_family,
            "repetition":rep,
            "n":RBF_N,
            "accuracy":result["accuracy"],
            "p_value":result["p_value"],
            "reject":result["reject_H0"],
            "n_test":result["n_test"],
            "n_permutations":result["n_permutations"]
        })

        if (rep+1)%20==0:
            print(f"  {rep+1}/{RBF_REPS}")

rbf_null_runs=pd.DataFrame(null_records)

print("Rows:",len(rbf_null_runs))
print("Expected:",4*RBF_REPS)

assert len(rbf_null_runs)==800


 template_tfidf
  20/200
  40/200
  60/200
  80/200
  100/200
  120/200
  140/200
  160/200
  180/200
  200/200

 timing_structural
  20/200
  40/200
  60/200
  80/200
  100/200
  120/200
  140/200
  160/200
  180/200
  200/200

 campaign_minilm
  20/200
  40/200
  60/200
  80/200
  100/200
  120/200
  140/200
  160/200
  180/200
  200/200

 op_minilm
  20/200
  40/200
  60/200
  80/200
  100/200
  120/200
  140/200
  160/200
  180/200
  200/200
Rows: 800
Expected: 800


In [10]:
# wilson confidence intervals
def wilson_interval(k,n,z=1.959963984540054):
    p=k/n
    denom=1+z**2/n
    center=(p+z**2/(2*n))/denom
    half=z*np.sqrt(
        p*(1-p)/n+z**2/(4*n**2)
    )/denom
    return center-half,center+half

In [11]:
rows=[]

for (scenario,severity),df in rbf_power_runs.groupby(
    ["scenario","severity"]
):
    k=int(df["reject"].sum())
    n=len(df)
    lo,hi=wilson_interval(k,n)

    rows.append({
        "scenario":scenario,
        "severity":severity,
        "repetitions":n,
        "rejections":k,
        "power":k/n,
        "ci95_lower":lo,
        "ci95_upper":hi,
        "mean_accuracy":df["accuracy"].mean(),
        "mean_p_value":df["p_value"].mean()
    })

rbf_power_summary=pd.DataFrame(rows)

display(
    rbf_power_summary.sort_values(
        ["scenario","severity"]
    )
)

,scenario,severity,repetitions,rejections,power,ci95_lower,ci95_upper,mean_accuracy,mean_p_value
0,Broken Timing,0.1,200,8,0.040,2.040563e-02,0.076932,0.51785,0.400693
1,Broken Timing,0.3,200,164,0.820,7.608852e-01,0.867054,0.56795,0.037871
2,Broken Timing,0.6,200,200,1.000,9.811547e-01,1.000000,0.66225,0.009901
3,Entity Inconsistency,0.1,200,0,0.000,1.734723e-18,0.018845,0.50425,0.806337
4,Entity Inconsistency,0.3,200,3,0.015,5.114238e-03,0.043166,0.51185,0.538614
5,Entity Inconsistency,0.6,200,36,0.180,1.329463e-01,0.239115,0.52740,0.275990
6,Missing Cross-Channel Reference,0.1,200,0,0.000,1.734723e-18,0.018845,0.50590,0.653168
7,Missing Cross-Channel Reference,0.3,200,16,0.080,4.984059e-02,0.125989,0.52100,0.366881
8,Missing Cross-Channel Reference,0.6,200,103,0.515,4.461085e-01,0.583326,0.55780,0.104307
9,Over-Polished Language,0.1,200,0,0.000,1.734723e-18,0.018845,0.50615,0.734109


In [12]:
# null summary
rows=[]

for null_family,df in rbf_null_runs.groupby("null_family"):
    k=int(df["reject"].sum())
    n=len(df)
    lo,hi=wilson_interval(k,n)

    rows.append({
        "null_family":null_family,
        "repetitions":n,
        "rejections":k,
        "type_I_error":k/n,
        "ci95_lower":lo,
        "ci95_upper":hi,
        "mean_accuracy":df["accuracy"].mean(),
        "mean_p_value":df["p_value"].mean()
    })

rbf_null_summary=pd.DataFrame(rows)

display(rbf_null_summary)

,null_family,repetitions,rejections,type_I_error,ci95_lower,ci95_upper,mean_accuracy,mean_p_value
0,campaign_minilm,200,5,0.025,0.010725,0.057178,0.49550,0.592624
1,op_minilm,200,9,0.045,0.023853,0.083297,0.50565,0.531485
2,template_tfidf,200,5,0.025,0.010725,0.057178,0.50105,0.564505
3,timing_structural,200,8,0.040,0.020406,0.076932,0.49950,0.559158


In [13]:
rbf_power_table=rbf_power_summary.pivot(
    index="scenario",
    columns="severity",
    values="power"
)

display(rbf_power_table)

severity,0.1,0.3,0.6
scenario,,,
Broken Timing,0.04,0.820,1.000
Entity Inconsistency,0.00,0.015,0.180
Missing Cross-Channel Reference,0.00,0.080,0.515
Over-Polished Language,0.00,0.185,0.725
Template Repetition,0.49,1.000,1.000


In [14]:
rbf_power_runs.to_csv(
    OUT_DIR/"c2st_rbf_n100_corrected_power_runs.csv",
    index=False
)

rbf_power_summary.to_csv(
    OUT_DIR/"c2st_rbf_n100_corrected_power_summary.csv",
    index=False
)

rbf_null_runs.to_csv(
    OUT_DIR/"c2st_rbf_n100_corrected_null_runs.csv",
    index=False
)

rbf_null_summary.to_csv(
    OUT_DIR/"c2st_rbf_n100_corrected_null_summary.csv",
    index=False
)

print("Saved to:",OUT_DIR)

Saved to: /Users/jessica/fidelity-metric-lab/results/week8/c2st_rbf_corrected


In [15]:
# old vs corrected RBF power table 
# Previous RBF-SVM n=100 anchor results
old_rbf_power=pd.DataFrame([
    ["Broken Timing",.10,.000],
    ["Broken Timing",.30,.000],
    ["Broken Timing",.60,.095],

    ["Entity Inconsistency",.10,.000],
    ["Entity Inconsistency",.30,.000],
    ["Entity Inconsistency",.60,.005],

    ["Missing Cross-Channel Reference",.10,.000],
    ["Missing Cross-Channel Reference",.30,.000],
    ["Missing Cross-Channel Reference",.60,.160],

    ["Over-Polished Language",.10,.000],
    ["Over-Polished Language",.30,.000],
    ["Over-Polished Language",.60,.275],

    ["Template Repetition",.10,.020],
    ["Template Repetition",.30,.985],
    ["Template Repetition",.60,1.000]
],columns=["scenario","severity","old_power"])

corrected=rbf_power_summary[
    ["scenario","severity","power","ci95_lower","ci95_upper","mean_accuracy","mean_p_value"]
].rename(columns={
    "power":"corrected_power",
    "ci95_lower":"corrected_ci95_lower",
    "ci95_upper":"corrected_ci95_upper",
    "mean_accuracy":"corrected_mean_accuracy",
    "mean_p_value":"corrected_mean_p_value"
})

rbf_comparison=old_rbf_power.merge(
    corrected,
    on=["scenario","severity"],
    how="inner"
)

rbf_comparison["delta_power"]=(
    rbf_comparison["corrected_power"]
    -rbf_comparison["old_power"]
)

display(rbf_comparison)

,scenario,severity,old_power,corrected_power,corrected_ci95_lower,corrected_ci95_upper,corrected_mean_accuracy,corrected_mean_p_value,delta_power
0,Broken Timing,0.1,0.000,0.040,2.040563e-02,0.076932,0.51785,0.400693,0.040
1,Broken Timing,0.3,0.000,0.820,7.608852e-01,0.867054,0.56795,0.037871,0.820
2,Broken Timing,0.6,0.095,1.000,9.811547e-01,1.000000,0.66225,0.009901,0.905
3,Entity Inconsistency,0.1,0.000,0.000,1.734723e-18,0.018845,0.50425,0.806337,0.000
4,Entity Inconsistency,0.3,0.000,0.015,5.114238e-03,0.043166,0.51185,0.538614,0.015
5,Entity Inconsistency,0.6,0.005,0.180,1.329463e-01,0.239115,0.52740,0.275990,0.175
6,Missing Cross-Channel Reference,0.1,0.000,0.000,1.734723e-18,0.018845,0.50590,0.653168,0.000
7,Missing Cross-Channel Reference,0.3,0.000,0.080,4.984059e-02,0.125989,0.52100,0.366881,0.080
8,Missing Cross-Channel Reference,0.6,0.160,0.515,4.461085e-01,0.583326,0.55780,0.104307,0.355
9,Over-Polished Language,0.1,0.000,0.000,1.734723e-18,0.018845,0.50615,0.734109,0.000


In [16]:
comparison_path=OUT_DIR/"c2st_rbf_old_vs_corrected.csv"

rbf_comparison.to_csv(
    comparison_path,
    index=False
)

print("Saved:",comparison_path)

Saved: /Users/jessica/fidelity-metric-lab/results/week8/c2st_rbf_corrected/c2st_rbf_old_vs_corrected.csv


In [17]:
old_rbf_null=pd.DataFrame([
    ["campaign_minilm",.040],
    ["op_minilm",.025],
    ["template_tfidf",.030],
    ["timing_structural",.050]
],columns=["null_family","old_type_I"])

corrected_null=rbf_null_summary[
    ["null_family","type_I_error","ci95_lower","ci95_upper"]
].rename(columns={
    "type_I_error":"corrected_type_I",
    "ci95_lower":"corrected_ci95_lower",
    "ci95_upper":"corrected_ci95_upper"
})

rbf_null_comparison=old_rbf_null.merge(
    corrected_null,
    on="null_family",
    how="inner"
)

rbf_null_comparison["delta_type_I"]=(
    rbf_null_comparison["corrected_type_I"]
    -rbf_null_comparison["old_type_I"]
)

display(rbf_null_comparison)

,null_family,old_type_I,corrected_type_I,corrected_ci95_lower,corrected_ci95_upper,delta_type_I
0,campaign_minilm,0.040,0.025,0.010725,0.057178,-0.015
1,op_minilm,0.025,0.045,0.023853,0.083297,0.020
2,template_tfidf,0.030,0.025,0.010725,0.057178,-0.005
3,timing_structural,0.050,0.040,0.020406,0.076932,-0.010


In [18]:
null_comparison_path=OUT_DIR/"c2st_rbf_old_vs_corrected_null.csv"

rbf_null_comparison.to_csv(
    null_comparison_path,
    index=False
)

print("Saved:",null_comparison_path)

Saved: /Users/jessica/fidelity-metric-lab/results/week8/c2st_rbf_corrected/c2st_rbf_old_vs_corrected_null.csv


After correcting the C2ST comparison so that Linear and RBF use the same paired train/test split and paired permutation significance test, the large performance gap observed previously largely disappeared. The corrected RBF-SVM closely matched the linear classifier for Broken Timing, Entity Inconsistency, and Missing Cross-Channel Reference, while differences remained for Over-Polished Language and low-severity Template Repetition. Therefore, the earlier conclusion that nonlinear C2ST was generally weaker was attributable in substantial part to a mismatch in the significance-testing procedure rather than classifier family alone.

| Representation | Type I |
|---|---:|
| Campaign MiniLM | .025 |
| Message MiniLM | .045 |
| TF-IDF | .025 |
| Structural | .040 |

They are all reasonably close to $\alpha = 0.05$, with no evidence of significant inflation.

### Anchor Pruning Table Construction 

In [3]:
# Paths
PREREG_DIR=RESULTS_DIR/"week8"/"preregistration"
PREREG_DIR.mkdir(parents=True,exist_ok=True)

RBF_DIR=RESULTS_DIR/"week8"/"c2st_rbf_corrected"

In [4]:
# LOAD 15 CORRECTED RBF
rbf_power=pd.read_csv(
    RBF_DIR/"c2st_rbf_n100_corrected_power_summary.csv"
)

rbf_null=pd.read_csv(
    RBF_DIR/"c2st_rbf_n100_corrected_null_summary.csv"
)

display(rbf_power.head())
display(rbf_null)

,scenario,severity,repetitions,rejections,power,ci95_lower,ci95_upper,mean_accuracy,mean_p_value
0,Broken Timing,0.1,200,8,0.040,2.040563e-02,0.076932,0.51785,0.400693
1,Broken Timing,0.3,200,164,0.820,7.608852e-01,0.867054,0.56795,0.037871
2,Broken Timing,0.6,200,200,1.000,9.811547e-01,1.000000,0.66225,0.009901
3,Entity Inconsistency,0.1,200,0,0.000,1.734723e-18,0.018845,0.50425,0.806337
4,Entity Inconsistency,0.3,200,3,0.015,5.114238e-03,0.043166,0.51185,0.538614


,null_family,repetitions,rejections,type_I_error,ci95_lower,ci95_upper,mean_accuracy,mean_p_value
0,campaign_minilm,200,5,0.025,0.010725,0.057178,0.49550,0.592624
1,op_minilm,200,9,0.045,0.023853,0.083297,0.50565,0.531485
2,template_tfidf,200,5,0.025,0.010725,0.057178,0.50105,0.564505
3,timing_structural,200,8,0.040,0.020406,0.076932,0.49950,0.559158


In [ ]:
# previous anchors
week7_power_rows=[]

week7_power={
    "Broken Timing":{
        .10:[.025,.975,.975,.330],
        .30:[.835,1.000,1.000,.940],
        .60:[1.000,1.000,1.000,1.000]
    },
    "Entity Inconsistency":{
        .10:[.000,.605,.640,.045],
        .30:[.025,1.000,1.000,.025],
        .60:[.165,1.000,1.000,.090]
    },
    "Missing Cross-Channel Reference":{
        .10:[.005,.140,.140,.125],
        .30:[.125,.550,.560,.310],
        .60:[.505,.995,.995,.685]
    },
    "Over-Polished Language":{
        .10:[.000,.870,.880,.130],
        .30:[.280,1.000,1.000,.025],
        .60:[.930,1.000,1.000,.040]
    },
    "Template Repetition":{
        .10:[.330,.950,.950,.970],
        .30:[1.000,1.000,1.000,1.000],
        .60:[1.000,1.000,1.000,1.000]
    }
}

week7_configs=[
    "C2ST-Linear",
    "Energy",
    "MMD",
    "Sinkhorn"
]

for scenario,severities in week7_power.items():
    for severity,powers in severities.items():
        for config,power in zip(week7_configs,powers):
            week7_power_rows.append({
                "Scenario":scenario,
                "Severity":severity,
                "Test Configuration":config,
                "Power @ n=100":power
            })

week7_power_df=pd.DataFrame(week7_power_rows)

print("Week 7 anchor rows:",len(week7_power_df))
display(week7_power_df.head())

Week 7 anchor rows: 60


,Scenario,Severity,Test Configuration,Power @ n=100
0,Broken Timing,0.1,C2ST-Linear,0.025
1,Broken Timing,0.1,Energy,0.975
2,Broken Timing,0.1,MMD,0.975
3,Broken Timing,0.1,Sinkhorn,0.330
4,Broken Timing,0.3,C2ST-Linear,0.835


In [6]:
# previous nulls
week7_null_rates={
    ("Broken Timing","C2ST-Linear"):.035,
    ("Broken Timing","Energy"):.065,
    ("Broken Timing","MMD"):.055,
    ("Broken Timing","Sinkhorn"):.045,

    ("Entity Inconsistency","C2ST-Linear"):.035,
    ("Entity Inconsistency","Energy"):.055,
    ("Entity Inconsistency","MMD"):.055,
    ("Entity Inconsistency","Sinkhorn"):.065,

    ("Missing Cross-Channel Reference","C2ST-Linear"):.035,
    ("Missing Cross-Channel Reference","Energy"):.055,
    ("Missing Cross-Channel Reference","MMD"):.055,
    ("Missing Cross-Channel Reference","Sinkhorn"):.065,

    ("Over-Polished Language","C2ST-Linear"):.050,
    ("Over-Polished Language","Energy"):.045,
    ("Over-Polished Language","MMD"):.050,
    ("Over-Polished Language","Sinkhorn"):.050,

    ("Template Repetition","C2ST-Linear"):.030,
    ("Template Repetition","Energy"):.045,
    ("Template Repetition","MMD"):.045,
    ("Template Repetition","Sinkhorn"):.045
}

In [7]:
# null threshold function 
def wilson_upper(k,n=200,z=1.959963984540054):
    p=k/n
    denom=1+z**2/n
    center=(p+z**2/(2*n))/denom
    half=z*np.sqrt(
        p*(1-p)/n+z**2/(4*n**2)
    )/denom
    return center+half

In [8]:
week7_power_df["Null rate"]=week7_power_df.apply(
    lambda row:week7_null_rates[
        (row["Scenario"],row["Test Configuration"])
    ],
    axis=1
)

week7_power_df["Null threshold"]=week7_power_df["Null rate"].apply(
    lambda rate:wilson_upper(round(rate*200),200)
)

In [9]:
rbf_df=rbf_power[
    ["scenario","severity","power"]
].rename(columns={
    "scenario":"Scenario",
    "severity":"Severity",
    "power":"Power @ n=100"
})

rbf_df["Test Configuration"]="C2ST-RBF"

In [10]:
# assign each scenario to their null family
scenario_to_null_family={
    "Broken Timing":"timing_structural",
    "Entity Inconsistency":"campaign_minilm",
    "Missing Cross-Channel Reference":"campaign_minilm",
    "Over-Polished Language":"op_minilm",
    "Template Repetition":"template_tfidf"
}

rbf_null_map=dict(
    zip(
        rbf_null["null_family"],
        rbf_null["type_I_error"]
    )
)

rbf_df["Null rate"]=rbf_df["Scenario"].map(
    lambda s:rbf_null_map[scenario_to_null_family[s]]
)

rbf_df["Null threshold"]=rbf_df["Null rate"].apply(
    lambda rate:wilson_upper(round(rate*200),200)
)

In [11]:
anchor_table=pd.concat(
    [week7_power_df,rbf_df],
    ignore_index=True
)

In [12]:
# apply pruning rule 
anchor_table["n=50 decision"]=np.where(
    anchor_table["Power @ n=100"]<=anchor_table["Null threshold"],
    "PRUNE",
    "RETAIN"
)

anchor_table["n=200 decision"]=np.where(
    np.isclose(anchor_table["Power @ n=100"],1.0),
    "PRUNE",
    "RETAIN"
)

In [13]:
# organize the table
scenario_order=[
    "Broken Timing",
    "Entity Inconsistency",
    "Missing Cross-Channel Reference",
    "Over-Polished Language",
    "Template Repetition"
]

config_order=[
    "C2ST-Linear",
    "C2ST-RBF",
    "Energy",
    "MMD",
    "Sinkhorn"
]

anchor_table["Scenario"]=pd.Categorical(
    anchor_table["Scenario"],
    categories=scenario_order,
    ordered=True
)

anchor_table["Test Configuration"]=pd.Categorical(
    anchor_table["Test Configuration"],
    categories=config_order,
    ordered=True
)

anchor_table=anchor_table.sort_values(
    ["Scenario","Severity","Test Configuration"]
).reset_index(drop=True)

In [14]:
anchor_pruning_table=anchor_table[
    [
        "Scenario",
        "Severity",
        "Test Configuration",
        "Power @ n=100",
        "Null threshold",
        "n=50 decision",
        "n=200 decision"
    ]
].copy()

display(anchor_pruning_table)

,Scenario,Severity,Test Configuration,Power @ n=100,Null threshold,n=50 decision,n=200 decision
0,Broken Timing,0.1,C2ST-Linear,0.025,0.070471,PRUNE,RETAIN
1,Broken Timing,0.1,C2ST-RBF,0.040,0.076932,PRUNE,RETAIN
2,Broken Timing,0.1,Energy,0.975,0.108019,RETAIN,RETAIN
3,Broken Timing,0.1,MMD,0.975,0.095787,RETAIN,RETAIN
4,Broken Timing,0.1,Sinkhorn,0.330,0.083297,RETAIN,RETAIN
...,...,...,...,...,...,...,...
70,Template Repetition,0.6,C2ST-Linear,1.000,0.063894,RETAIN,PRUNE
71,Template Repetition,0.6,C2ST-RBF,1.000,0.057178,RETAIN,PRUNE
72,Template Repetition,0.6,Energy,1.000,0.083297,RETAIN,PRUNE
73,Template Repetition,0.6,MMD,1.000,0.083297,RETAIN,PRUNE


In [15]:
# validation
print("Rows:",len(anchor_pruning_table))

n50_retained=(anchor_pruning_table["n=50 decision"]=="RETAIN").sum()
n200_retained=(anchor_pruning_table["n=200 decision"]=="RETAIN").sum()

print("Retained n=50:",n50_retained)
print("Retained n=200:",n200_retained)
print("Total retained extension cells:",n50_retained+n200_retained)

assert len(anchor_pruning_table)==75
assert n50_retained+n200_retained==110

Rows: 75
Retained n=50: 60
Retained n=200: 50
Total retained extension cells: 110


In [16]:
out_path=PREREG_DIR/"n100_anchor_pruning_table.csv"

anchor_pruning_table.to_csv(
    out_path,
    index=False
)

print("Saved:",out_path)

Saved: /Users/jessica/fidelity-metric-lab/results/week8/preregistration/n100_anchor_pruning_table.csv


*Note = "Week 7" results were carried out during week 6. The name "Week 7" was kept througout this notebook because the results were already stored under that name.*